In [ ]:
# CELL 1: GOOGLE COLAB SETUP & DRIVE MOUNT (OPTIONAL)
from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# CELL 2: CONFIGURATION & FILE PATHS
# Provide your Google Colab CSV paths below:
TRAIN_DATASET_PATH = "/content/PASTE_TRAIN_DATASET_HERE.csv"

# List of multiple test files to evaluate:
TEST_DATASET_PATHS = [
    "/content/PASTE_TEST_DATASET_1_HERE.csv",
    "/content/PASTE_TEST_DATASET_2_HERE.csv"
]

SOM_MODEL_PATH = "/content/som_trained_model.npz"


In [ ]:
# CELL 3: IMPORTS & HELPER FUNCTIONS
import numpy as np
import matplotlib.pyplot as plt
import os

def load_som_csv(file_path):
    """
    Reads CSV. Separates numerical features from optional last-column labels.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    features = []
    labels = []

    with open(file_path, "r") as f:
        lines = f.readlines()

    first_line = lines[0].strip()
    delimiter = "\t" if "\t" in first_line else ","
    start_idx = 0

    tokens = first_line.split(delimiter)
    try:
        float(tokens[0])
    except ValueError:
        start_idx = 1  # Skip header

    has_labels = False
    for line in lines[start_idx:]:
        line = line.strip()
        if not line:
            continue
        parts = line.split(delimiter)

        try:
            float(parts[-1])
            row_feat = [float(p.strip()) for p in parts]
            row_label = "Unknown"
        except ValueError:
            row_feat = [float(p.strip()) for p in parts[:-1]]
            row_label = parts[-1].strip()
            has_labels = True

        features.append(row_feat)
        labels.append(row_label)

    return np.array(features, dtype=np.float64), labels, has_labels

def find_bmu(sample, weights, grid_rows, grid_cols):
    """
    Finds Best Matching Unit (BMU): neuron whose weight vector
    has minimum Euclidean distance to the sample.
    """
    min_dist = float("inf")
    bmu_coord = (0, 0)

    for r in range(grid_rows):
        for c in range(grid_cols):
            w = weights[r, c]
            dist = np.sum((sample - w) ** 2)
            if dist < min_dist:
                min_dist = dist
                bmu_coord = (r, c)

    return bmu_coord, np.sqrt(min_dist)


In [ ]:
# CELL 4: MODULE 1 - TRAINING MODULE (UNSUPERVISED SOM CLUSTERING)
def train_som():
    print("=" * 60)
    print("MODULE 1: TRAINING KOHONEN SELF-ORGANIZING MAP (SOM)")
    print("=" * 60)

    # 1. Read Training Dataset
    print(f"Reading training data from: {TRAIN_DATASET_PATH}")
    X_train, y_train, has_labels = load_som_csv(TRAIN_DATASET_PATH)
    num_samples, num_features = X_train.shape
    print(f"Loaded {num_samples} samples with {num_features} features.")

    # 2. Min-Max Normalization
    feat_min = np.min(X_train, axis=0)
    feat_max = np.max(X_train, axis=0)
    feat_diff = np.where((feat_max - feat_min) == 0, 1.0, feat_max - feat_min)
    X_train_norm = (X_train - feat_min) / feat_diff

    # 3. Grid Configuration
    grid_rows, grid_cols = 5, 5
    initial_learning_rate = 0.5
    initial_radius = max(grid_rows, grid_cols) / 2.0
    total_epochs = 150

    np.random.seed(42)
    weights = np.random.uniform(0.0, 1.0, (grid_rows, grid_cols, num_features))
    quantization_errors = []

    # 4. Training Loop
    for epoch in range(total_epochs):
        # Exponential Decay of LR and Neighborhood Radius
        lr = initial_learning_rate * np.exp(-epoch / total_epochs)
        sigma = initial_radius * np.exp(-epoch / total_epochs)
        sigma_sq_2 = 2.0 * (sigma ** 2)

        epoch_dist_sum = 0.0

        for i in range(num_samples):
            sample = X_train_norm[i]

            # Find BMU
            (bmu_r, bmu_c), bmu_dist = find_bmu(sample, weights, grid_rows, grid_cols)
            epoch_dist_sum += bmu_dist

            # Update BMU and neighbors on the 2D lattice
            for r in range(grid_rows):
                for c in range(grid_cols):
                    grid_dist_sq = (r - bmu_r) ** 2 + (c - bmu_c) ** 2
                    if grid_dist_sq < (3 * sigma) ** 2:
                        neighborhood = np.exp(-grid_dist_sq / sigma_sq_2)
                        weights[r, c] += lr * neighborhood * (sample - weights[r, c])

        avg_qe = epoch_dist_sum / num_samples
        quantization_errors.append(avg_qe)

        if (epoch + 1) % 25 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:3d}/{total_epochs} | LR: {lr:.4f} | Radius: {sigma:.2f} | Quantization Error: {avg_qe:.4f}")

    print(f"\nFinal Quantization Error: {quantization_errors[-1]:.4f}")

    # 5. Save Model Parameters
    np.savez(
        SOM_MODEL_PATH,
        weights=weights,
        grid_rows=grid_rows,
        grid_cols=grid_cols,
        feat_min=feat_min,
        feat_diff=feat_diff
    )
    print(f"Trained SOM parameters saved to: {SOM_MODEL_PATH}")

    # 6. Plot Quantization Error Curve
    plt.figure(figsize=(7, 4))
    plt.plot(range(1, total_epochs + 1), quantization_errors, color="green")
    plt.xlabel("Epoch")
    plt.ylabel("Average Quantization Error")
    plt.title("SOM: Epoch vs Quantization Error")
    plt.grid(True)
    plt.show()

# Run Training
train_som()


In [ ]:
# CELL 5: MODULE 2 - TESTING MODULE (CLUSTER MAPPING)
def test_som():
    print("=" * 60)
    print("MODULE 2: TESTING KOHONEN SOM (CLUSTER MAPPING)")
    print("=" * 60)

    if not os.path.exists(SOM_MODEL_PATH):
        raise FileNotFoundError(f"Model {SOM_MODEL_PATH} not found. Please run training cell first.")

    # 1. Load Trained Parameters
    model_data = np.load(SOM_MODEL_PATH)
    weights = model_data["weights"]
    grid_rows = int(model_data["grid_rows"])
    grid_cols = int(model_data["grid_cols"])
    feat_min = model_data["feat_min"]
    feat_diff = model_data["feat_diff"]

    print(f"Loaded SOM Grid Dimensions: ({grid_rows} x {grid_cols})")

    # 2. Iterate Over Multiple Test Files
    for file_idx, test_path in enumerate(TEST_DATASET_PATHS, start=1):
        print("\n" + "-" * 50)
        print(f"Testing File {file_idx}: {test_path}")
        print("-" * 50)

        if not os.path.exists(test_path):
            print(f"Warning: File {test_path} not found. Skipping.")
            continue

        X_test, y_test, has_labels = load_som_csv(test_path)
        X_test_norm = (X_test - feat_min) / feat_diff

        cluster_counts = np.zeros((grid_rows, grid_cols), dtype=int)
        sample_assignments = []

        # 3. Assign each sample to its BMU
        for i in range(len(X_test_norm)):
            (bmu_r, bmu_c), dist = find_bmu(X_test_norm[i], weights, grid_rows, grid_cols)
            cluster_counts[bmu_r, bmu_c] += 1
            cluster_id = f"Cluster_({bmu_r},{bmu_c})"
            sample_assignments.append((cluster_id, y_test[i] if has_labels else "N/A"))

        print(f"Total Test Samples: {len(X_test_norm)}")
        print("\nNeuron Activation / Hit Count Grid (2D Frequency Distribution):")
        print(cluster_counts)

        print("\nFirst 5 Sample Cluster Assignments:")
        print(f"{'Sample #':<10} | {'Assigned BMU Cluster':<22} | {'True Label (If Present)':<20}")
        print("-" * 60)
        for i in range(min(5, len(sample_assignments))):
            c_id, lbl = sample_assignments[i]
            print(f"{i+1:<10} | {c_id:<22} | {lbl:<20}")

        # 4. Display 2D Hit Heatmap
        plt.figure(figsize=(6, 5))
        plt.imshow(cluster_counts, cmap="YlGnBu", interpolation="nearest")
        plt.colorbar(label="Number of Samples Mapped")
        plt.title(f"SOM Cluster Activation Heatmap (Test File {file_idx})")
        plt.xlabel("Neuron Column")
        plt.ylabel("Neuron Row")
        for r in range(grid_rows):
            for c in range(grid_cols):
                plt.text(c, r, str(cluster_counts[r, c]), ha="center", va="center", color="red", fontweight="bold")
        plt.show()

# Run Testing
test_som()


## 3. Important Viva Questions & Answers
1. **Q: Is Kohonen SOM a supervised or unsupervised algorithm?**  
   *Ans:* It is strictly unsupervised. The network self-organizes without using target class labels.
2. **Q: What is a Best Matching Unit (BMU)?**  
   *Ans:* The neuron whose weight vector has the minimum Euclidean distance to the given input vector.
3. **Q: Why do the learning rate and neighborhood radius decay over time?**  
   *Ans:* Initial large values allow broad global topological ordering across the data space. As they decay, the map stabilizes and performs fine local clustering.
4. **Q: What is Quantization Error (QE)?**  
   *Ans:* The average Euclidean distance between each data sample and its BMU weight vector. Lower QE indicates better codebook representation.
